# Rothermel surface-fire rate of spread
## Scott & Burgan (2005) standard fire behavior fuel models

Pick one of the 40 standard fuel models, set the environmental inputs
(fuel moisture, midflame wind, slope), and read off the **rate of spread (ROS)**.

The calculation runs in Rothermel's original *imperial* units so the intermediate
parameters (characteristic SAV, packing ratio, reaction intensity, ...) match the
published tables; the final ROS is converted to metric at the end.

**References**

* Rothermel, R.C. 1972. *A mathematical model for predicting fire spread in wildland fuels.* INT-115.
* Albini, F.A. 1976. *Estimating wildfire behavior and effects.* INT-GTR-30.
* Scott, J.H.; Burgan, R.E. 2005. *Standard fire behavior fuel models.* RMRS-GTR-153.
  (the "FBFM40" set; frequently mis-cited as *Scott & Burgess*).
* Andrews, P.L. 2018. *The Rothermel surface fire spread model ...* RMRS-GTR-371. (equation listing)

> **Note** Four transcription errors in the source fuel-model JSON are corrected in
> `rothermel.py` (`_CORRECTIONS`): SH1/SH6/TL3 100-h loads and the SH9 fuel-bed depth.
> The implementation reproduces the published characteristic SAV and packing ratio for
> all 40 models (run `python validate.py`).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

from rothermel import (
    load_fuel_models, compute_ros, Moisture,
    mps_to_ft_min, kmh_to_ft_min, deg_to_slope_fraction,
    wind_adjustment_factor, wind_10m_to_20ft,
    TONS_PER_ACRE_TO_LB_PER_FT2 as T2LB,
)

MODELS = load_fuel_models()           # 40 burnable models, keyed by code
print(f'Loaded {len(MODELS)} standard fuel models.')

## Inputs

Wind is **midflame** wind speed (the wind acting on the flames, typically a fraction
of the 10 m open wind). Slope is the terrain angle in degrees. Live herbaceous moisture
drives the *dynamic* load transfer (curing) for the grass/grass-shrub models.

In [ ]:
# ----- fuel model selector (grouped by category) -----
options = [(f'{c}  -  {MODELS[c].name}', c) for c in MODELS]
fuel_dd = widgets.Dropdown(options=options, value='GR2', description='Fuel model',
                           layout=widgets.Layout(width='520px'),
                           style={'description_width': '120px'})

def _slider(desc, lo, hi, val, step, unit):
    return widgets.FloatSlider(description=desc, min=lo, max=hi, value=val, step=step,
                               continuous_update=False, readout_format='.0f',
                               layout=widgets.Layout(width='420px'),
                               style={'description_width': '150px'})

m_1h    = _slider('1-h dead moist %',    1, 40,  6, 1, '%')
m_10h   = _slider('10-h dead moist %',   1, 40,  7, 1, '%')
m_100h  = _slider('100-h dead moist %',  1, 50,  8, 1, '%')
m_herb  = _slider('live herb moist %',  30, 300, 90, 5, '%')
m_woody = _slider('live woody moist %', 30, 300, 90, 5, '%')

wind    = widgets.FloatText(value=5.0, description='Wind speed',
                            style={'description_width': '150px'},
                            layout=widgets.Layout(width='300px'))
wind_unit = widgets.Dropdown(options=['m/s', 'km/h', 'mi/h', 'ft/min'], value='m/s',
                             layout=widgets.Layout(width='90px'))
# what the entered wind refers to; midflame = used directly (no WAF applied)
wind_ref = widgets.Dropdown(
    options=[('measured at 10 m (open)', '10m'),
             ('measured at 20 ft', '20ft'),
             ('midflame (use directly)', 'midflame')],
    value='10m', description='Wind reference',
    style={'description_width': '150px'}, layout=widgets.Layout(width='420px'))
slope   = _slider('slope (deg)', 0, 50, 0, 1, 'deg')
wind_limit = widgets.Checkbox(value=False, description='Apply Rothermel wind limit (0.9 I_R)',
                              indent=False)

# ----- wind adjustment factor (20-ft -> midflame) -----
waf_mode = widgets.Dropdown(
    options=[('auto from canopy (sheltered if crown-fill > 5%)', 'auto'),
             ('unsheltered (fuel-bed depth only)', 'unsheltered'),
             ('enter WAF directly', 'direct')],
    value='unsheltered', description='WAF model',
    style={'description_width': '150px'}, layout=widgets.Layout(width='480px'))
canopy_cover  = _slider('canopy cover %',   0, 100, 40, 5, '%')
canopy_height = _slider('canopy height (ft)', 0, 150, 50, 5, 'ft')
crown_ratio   = _slider('crown ratio %',    0, 100, 70, 5, '%')
waf_direct    = widgets.FloatText(value=0.30, description='WAF (direct)',
                                  style={'description_width': '150px'},
                                  layout=widgets.Layout(width='280px'))

ui = widgets.VBox([
    fuel_dd,
    widgets.HTML('<b>Dead fuel moisture</b>'), m_1h, m_10h, m_100h,
    widgets.HTML('<b>Live fuel moisture</b>'), m_herb, m_woody,
    widgets.HTML('<b>Wind &amp; slope</b>'),
    widgets.HBox([wind, wind_unit]), wind_ref, slope, wind_limit,
    widgets.HTML('<b>Wind adjustment factor</b> (ignored if wind reference is midflame)'),
    waf_mode, canopy_cover, canopy_height, crown_ratio, waf_direct,
])

## Rate of spread
Adjust any control above and the result below updates automatically.

In [ ]:
import math

_WIND_CONV = {'m/s': mps_to_ft_min, 'km/h': kmh_to_ft_min,
              'mi/h': lambda v: v * 88.0, 'ft/min': lambda v: v}

# imperial -> SI conversions for the intermediate parameters
IR_BTUFT2MIN_TO_KWM2 = 0.189266    # BTU/ft^2/min -> kW/m^2  (J/m^2/s)
SAV_PERFT_TO_PERM    = 3.280840    # ft^-1 -> m^-1
HEAT_BTUFT3_TO_KJM3  = 37.2589     # BTU/ft^3 -> kJ/m^3
MS_TO_FT_MIN         = 196.850394  # 1 m/s -> ft/min

out = widgets.Output()

def _midflame(fm):
    """Return (midflame_ft_min, info_str) applying the chosen WAF chain."""
    speed_ftmin = _WIND_CONV[wind_unit.value](wind.value)   # at its reference height
    if wind_ref.value == 'midflame':
        return speed_ftmin, 'entered as midflame wind (no WAF applied)'
    # reduce to 20-ft open wind first
    w20 = wind_10m_to_20ft(speed_ftmin) if wind_ref.value == '10m' else speed_ftmin
    if waf_mode.value == 'direct':
        waf, regime = waf_direct.value, 'user-entered'
    else:
        cc = canopy_cover.value / 100 if waf_mode.value == 'auto' else 0.0
        res = wind_adjustment_factor(fm.depth, canopy_cover=cc,
                                     canopy_height_ft=canopy_height.value,
                                     crown_ratio=crown_ratio.value / 100)
        waf, regime = res.waf, res.regime
    ref = '10-m' if wind_ref.value == '10m' else '20-ft'
    info = f'WAF = {waf:.2f} ({regime}); {ref} wind reduced to midflame'
    return w20 * waf, info

def _render(*_):
    fm = MODELS[fuel_dd.value]
    moist = Moisture.from_percent(m_1h.value, m_10h.value, m_100h.value,
                                  m_herb.value, m_woody.value)
    wind_ftmin, waf_info = _midflame(fm)
    r = compute_ros(fm, moist, wind_ftmin, deg_to_slope_fraction(slope.value),
                    apply_wind_limit=wind_limit.value)
    midflame_ms = wind_ftmin / MS_TO_FT_MIN

    # wind-factor coefficients (Rothermel 1972; sigma in ft^-1, U in ft/min)
    sigma = r.characteristic_sav
    c_w = 7.47 * math.exp(-0.133 * sigma ** 0.55)
    b_w = 0.02526 * sigma ** 0.54
    e_w = 0.715 * math.exp(-3.59e-4 * sigma)
    # lump the sigma/packing terms:  phi_w = K * U^B  with K = C * beta_rel^(-E)
    u_factor = MS_TO_FT_MIN ** b_w          # 196.85^B, the wind-unit conversion
    k_imp = c_w * r.relative_packing_ratio ** (-e_w)   # K with U in ft/min
    k_si = k_imp * u_factor                            # K with U in m/s

    def t2(x):   # lb/ft^2 -> t/acre for display
        return x / T2LB
    loads = (f'1-h {t2(fm.w_1h):.2f}, 10-h {t2(fm.w_10h):.2f}, 100-h {t2(fm.w_100h):.2f}, '
             f'herb {t2(fm.w_herb):.2f}, woody {t2(fm.w_woody):.2f}  t/ac')
    warn = ''
    if r.wind_limited:
        warn = ('\n> ?? **Wind-limited**: midflame wind exceeded the Rothermel 0.9&middot;I_R '
                'reliability limit and was capped. Uncheck the wind limit to see the raw value.')

    md = f'''
### {fm.code} - {fm.name}  {'(dynamic)' if fm.dynamic else '(static)'}

| Rate of spread | |
|---|---|
| **{r.ros_m_min:.2f} m/min** | {r.ros_km_h:6.3f} km/h |
| {r.ros_m_s:8.4f} m/s | {r.ros_ft_min:8.2f} ft/min |
| {r.ros_ft_min/1.1:8.1f} chains/h | |

*{fm.description}*

**Fuel bed** loads: {loads} &nbsp;|&nbsp; depth {fm.depth} ft &nbsp;|&nbsp; dead M_x {fm.mx_dead*100:.0f}%

**Wind** midflame {midflame_ms:.2f} m/s ({wind_ftmin:.0f} ft/min) &nbsp;|&nbsp; {waf_info}

| Intermediate | imperial | SI (J, m, s) |
|---|---:|---:|
| Reaction intensity I_R | {r.reaction_intensity:,.0f} BTU/ft^2/min | {r.reaction_intensity*IR_BTUFT2MIN_TO_KWM2:,.1f} kW/m^2 |
| Characteristic SAV | {r.characteristic_sav:,.0f} ft^-1 | {r.characteristic_sav*SAV_PERFT_TO_PERM:,.0f} m^-1 |
| Packing ratio beta | {r.packing_ratio:.5f} | {r.packing_ratio:.5f} (-) |
| Relative packing ratio | {r.relative_packing_ratio:.2f} | {r.relative_packing_ratio:.2f} (-) |
| Propagating flux ratio xi | {r.propagating_flux_ratio:.4f} | {r.propagating_flux_ratio:.4f} (-) |
| Wind factor phi_w | {r.wind_factor:.2f} | {r.wind_factor:.2f} (-) |
| Slope factor phi_s | {r.slope_factor:.2f} | {r.slope_factor:.2f} (-) |
| Heat sink rho_b eps Q_ig | {r.heat_sink:.1f} BTU/ft^3 | {r.heat_sink*HEAT_BTUFT3_TO_KJM3:,.0f} kJ/m^3 |
| Live moisture of extinction | {r.live_mx*100:.0f}% | {r.live_mx*100:.0f}% |
| Herb cured fraction | {r.cured_fraction:.2f} | {r.cured_fraction:.2f} (-) |

**Rate of spread** combines the rows above (Rothermel 1972, eq. 52):

R = I_R &middot; xi &middot; (1 + phi_w + phi_s) / (rho_b eps Q_ig)

= {r.reaction_intensity:,.0f} &middot; {r.propagating_flux_ratio:.4f} &middot; (1 + {r.wind_factor:.2f} + {r.slope_factor:.2f}) / {r.heat_sink:.1f} = {r.ros_ft_min:.2f} ft/min &rarr; {r.ros_m_min:.2f} m/min

The numerator I_R&middot;xi&middot;(1+phi_w+phi_s) is the heat flux propagating into the unburnt fuel: the reaction intensity I_R scaled by the no-wind, no-slope propagating flux ratio xi, then amplified by the wind (phi_w) and slope (phi_s) factors. The denominator rho_b&middot;eps&middot;Q_ig is the heat sink, the energy needed to bring that fuel to ignition. Their ratio is the rate of spread shown at the top.

**Wind factor** phi_w = C &middot; U^B &middot; beta_rel^(-E) &nbsp; (Rothermel 1972; coefficients fit with sigma in ft^-1, U in ft/min)

| phi_w term | imperial (U in ft/min) | SI (U in m/s) |
|---|---:|---:|
| Characteristic SAV sigma | {sigma:,.0f} ft^-1 | {sigma*SAV_PERFT_TO_PERM:,.0f} m^-1 |
| Midflame wind U | {wind_ftmin:,.0f} ft/min | {midflame_ms:.3f} m/s |
| C = 7.47 exp(-0.133 sigma^0.55) | {c_w:.4f} | {c_w:.4f} |
| B = 0.02526 sigma^0.54 | {b_w:.4f} | {b_w:.4f} (-) |
| E = 0.715 exp(-3.59e-4 sigma) | {e_w:.4f} | {e_w:.4f} (-) |
| K = C &middot; beta_rel^(-E) | {k_imp:.4f} | {k_si:.4f} = K &middot; 196.85^B ({u_factor:.2f}) |
| Wind factor phi_w | {r.wind_factor:.3f} | {r.wind_factor:.3f} (-) |

*B and E are dimensionless and unchanged between systems. Collecting the sigma/packing terms gives phi_w = K &middot; U^B with K = C &middot; beta_rel^(-E). Only K carries the wind-speed unit: because U appears as U^B, switching U from ft/min to m/s multiplies K by 196.85^B, so phi_w = K_si &middot; U[m/s]^B reproduces the same value while C keeps its original value.*
{warn}
'''
    with out:
        out.clear_output(wait=True)
        display(Markdown(md))

_controls = (fuel_dd, m_1h, m_10h, m_100h, m_herb, m_woody, wind, wind_unit,
             wind_ref, slope, wind_limit, waf_mode, canopy_cover, canopy_height,
             crown_ratio, waf_direct)
for w in _controls:
    w.observe(_render, names='value')

_render()
display(ui, out)

## Notes

* **Midflame wind** is not the 10 m weather-station wind. Set the *wind reference* and a
  *WAF model* to convert 10-m or 20-ft wind to midflame wind (Albini & Baughman 1979;
  Andrews 2012, RMRS-GTR-266): unsheltered uses fuel-bed depth; sheltered uses canopy
  cover, height, and crown ratio (crown-fill portion f = (CC/3)&middot;CR, sheltered when
  f > 5%). 10-m open wind is taken as 1.15&times; the 20-ft wind.
* The **wind limit** caps the wind factor at the wind speed where Rothermel's data became
  unreliable (0.9 &middot; I_R). Cruz & Finney (2013) later revised this; toggle it off to compare.
* **Dynamic models** (all GR/GS, SH1, SH9, TU1, TU3) transfer part of the live herbaceous
  load to dead as it cures: fully green at >=120% moisture, fully cured at <=30%.
* `compute_ros` returns the full `SpreadResult`; use it directly for batch runs or plots.

## Fuel burnout time scale (LES coupling)

Rothermel above is **quasi-steady**: it returns a rate of spread and a reaction
intensity `I_R`, but it has no clock — it never tells you how long a point keeps
burning after the front passes. An atmospheric LES needs exactly that: a
time-resolved surface **heat flux** as its lower boundary condition. WRF-SFIRE
supplies it with a separate fuel mass-loss model layered on top of the spread
model; `burnout.py` reproduces that layer.

For a cell that ignites at `t_i`, the remaining dry-fuel fraction decays
exponentially with an e-folding (burnout) time `T_f`:

F(t) = exp(-(t - t_i) / T_f) &nbsp;&nbsp;→&nbsp;&nbsp; Q(t) = (E_net / T_f) · exp(-(t - t_i) / T_f)  [W/m²]

where `E_net` is the net heat per unit ground area: the dry load × heat of
combustion (8000 BTU/lb), minus the energy spent vaporising the fuel moisture.

`T_f` is **not** in Rothermel. The default closure here is the Anderson (1969)
flame residence time `τ = 384/σ` minutes, with `σ` the characteristic SAV
(ft⁻¹) taken straight from the `SpreadResult` — finer fuel burns faster, the
same SAV dependence Rothermel uses internally. Alternatively pass an explicit
`t_f` (s) or a WRF-SFIRE `wrf_weight` (converted as `weight/0.85`).

The survey below uses the slider defaults for moisture. `peak flux` is `Q(t_i)`,
the instantaneous flux right after ignition; the total energy under each
exponential equals `net heat`.

In [ ]:
from burnout import compute_burnout, anderson_residence_time

# nominal moisture for the survey (same defaults as the sliders above)
moist_tbl = Moisture.from_percent(6, 7, 8, 90, 90)

rows = []
for code, fm in MODELS.items():
    r = compute_ros(fm, moist_tbl, 0.0, 0.0)                 # sigma is wind-independent
    b = compute_burnout(fm, moist_tbl, characteristic_sav=r.characteristic_sav)
    rows.append((code, fm.name,
                 r.characteristic_sav * SAV_PERFT_TO_PERM,   # sigma in m^-1
                 b.t_f, b.load_dry, b.energy_net_MJ, b.peak_flux_kW))

header = ('| Fuel | Name | &sigma; (m&#8315;&#185;) | T_f Anderson (s) | dry load (kg/m&#178;) '
          '| net heat (MJ/m&#178;) | peak flux (kW/m&#178;) |\n'
          '|---|---|---:|---:|---:|---:|---:|')
body = '\n'.join(
    f'| {c} | {n} | {sav:,.0f} | {tf:,.1f} | {ld:.2f} | {e:,.1f} | {pf:,.0f} |'
    for (c, n, sav, tf, ld, e, pf) in rows)
display(Markdown('### Burnout time scale and heat release by fuel model\n\n'
                 + header + '\n' + body))